In [1]:
#!/usr/bin/env python
# coding: utf-8

# --------------------------------------------------------------------------- #
# Imports
# --------------------------------------------------------------------------- #
import os
import sys
import warnings
import datetime as dt
from datetime import date
import itertools
from typing import List, Tuple, Dict, Optional

import pandas as pd
import yfinance as yf
import requests
from tqdm import tqdm

# Sentiment
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Forecasting
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.statespace import sarimax

# --------------------------------------------------------------------------- #
# Global constants / configuration
# --------------------------------------------------------------------------- #
# Maximum number of tickers to process (limits runtime & API usage)
MAX_TICKERS = 20

# Number of days to forecast
FORECAST_DAYS = 7

# Default number of news articles to fetch
DEFAULT_NEWS_COUNT = 20

# Sentiment thresholds (VADER compound score)
POSITIVE_THRESH = 0.05
NEGATIVE_THRESH = -0.05

# NewsAPI – pick up the key from an environment variable
NEWSAPI_KEY = os.getenv("NEWSAPI_KEY")
if not NEWSAPI_KEY:
    print("❗️  ERROR: Please set the environment variable NEWSAPI_KEY with your NewsAPI.org key.")
    sys.exit(1)

# Silence some harmless warnings from statsmodels
warnings.filterwarnings("ignore", category=UserWarning)

# --------------------------------------------------------------------------- #
# Helper / utility functions
# --------------------------------------------------------------------------- #

def get_sp500_constituents() -> pd.DataFrame:
    """
    Return a DataFrame with at least the columns:
        Symbol, Security, GICS Sector, GICS Sub‑Industry

    The function tries multiple sources and normalizes column names dynamically.
    """
    import pandas as pd

    # List of data sources to try
    sources = [
        {
            "name": "Wikipedia",
            "url": "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
            "method": "html"
        },
        {
            "name": "GitHub CSV (datasets)",
            "url": "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv",
            "method": "csv"
        },
        {
            "name": "GitHub CSV (alternative)",
            "url": "https://raw.githubusercontent.com/fja05680/sp500/master/S%26P%20500%20Historical%20Components%20%26%20Changes(03-01-2023).csv",
            "method": "csv"
        }
    ]

    for source in sources:
        try:
            print(f"🔍  Trying {source['name']}...")
            
            if source["method"] == "html":
                tables = pd.read_html(source["url"], header=0)
                if not tables:
                    continue
                df = tables[0].copy()
            else:  # CSV
                df = pd.read_csv(source["url"])
            
            if df.empty:
                continue
                
            # ----------------------------------------------------------- #
            # Normalize column names dynamically
            # ----------------------------------------------------------- #
            # First, clean up unicode characters in column names
            df.columns = (
                df.columns.astype(str)
                         .str.replace("–", "-", regex=False)   # EN‑dash
                         .str.replace("‑", "-", regex=False)   # non‑breaking hyphen
                         .str.replace("\u2011", "-", regex=False)  # extra safety
                         .str.strip()
            )
            
            # Build smart column mapping
            rename_map = {}
            found_columns = {"Symbol": False, "Security": False, "GICS Sector": False, "GICS Sub‑Industry": False}
            
            for col in df.columns:
                col_lower = col.lower()
                
                # Symbol/Ticker detection
                if not found_columns["Symbol"] and any(x in col_lower for x in ["symbol", "ticker"]):
                    rename_map[col] = "Symbol"
                    found_columns["Symbol"] = True
                
                # Security/Company name detection  
                elif not found_columns["Security"] and any(x in col_lower for x in ["security", "company", "name"]) and "sub" not in col_lower:
                    rename_map[col] = "Security"
                    found_columns["Security"] = True
                
                # Sector detection
                elif not found_columns["GICS Sector"] and "sector" in col_lower and "sub" not in col_lower:
                    rename_map[col] = "GICS Sector"
                    found_columns["GICS Sector"] = True
                
                # Sub-industry detection (more flexible matching)
                elif not found_columns["GICS Sub‑Industry"] and (
                    ("sub" in col_lower and "industry" in col_lower) or
                    "subindustry" in col_lower.replace(" ", "").replace("-", "").replace("_", "")
                ):
                    rename_map[col] = "GICS Sub‑Industry"
                    found_columns["GICS Sub‑Industry"] = True
            
            # Apply the renaming
            df.rename(columns=rename_map, inplace=True)
            
            # If we don't have a sub-industry column, create a placeholder one
            if "GICS Sub‑Industry" not in df.columns and "GICS Sector" in df.columns:
                df["GICS Sub‑Industry"] = df["GICS Sector"]  # Use sector as fallback
                found_columns["GICS Sub‑Industry"] = True
                print("⚠️  No sub-industry column found, using sector as fallback")
            
            # Check if we have the minimum required columns
            required = {"Symbol", "Security", "GICS Sector"}  # Relaxed requirements
            available = set(df.columns)
            
            if required.issubset(available):
                # Normalize the ticker symbols (`.` → `-` for yfinance)
                df["Symbol"] = df["Symbol"].astype(str).str.replace(".", "-", regex=False)
                
                # Ensure we have sub-industry even if empty
                if "GICS Sub‑Industry" not in df.columns:
                    df["GICS Sub‑Industry"] = "Unknown"
                
                # Clean up the dataframe
                df = df.dropna(subset=["Symbol"]).reset_index(drop=True)
                
                print(f"✅  Successfully loaded {len(df)} tickers from {source['name']}")
                return df
            else:
                missing = required - available
                print(f"⚠️  {source['name']} missing required columns: {missing}")
                continue
                
        except Exception as e:
            print(f"⚠️  {source['name']} failed: {e}")
            continue
    
    # ------------------------------------------------------------------- #
    # Final fallback: Create a minimal dataset with major tickers
    # ------------------------------------------------------------------- #
    print("⚠️  All external sources failed, using built-in fallback data...")
    
    fallback_data = [
        # Technology
        ("AAPL", "Apple Inc.", "Information Technology", "Technology Hardware, Storage & Peripherals"),
        ("MSFT", "Microsoft Corporation", "Information Technology", "Systems Software"),
        ("GOOGL", "Alphabet Inc.", "Communication Services", "Interactive Media & Services"),
        ("AMZN", "Amazon.com Inc.", "Consumer Discretionary", "Internet & Direct Marketing Retail"),
        ("NVDA", "NVIDIA Corporation", "Information Technology", "Semiconductors & Semiconductor Equipment"),
        ("META", "Meta Platforms Inc.", "Communication Services", "Interactive Media & Services"),
        ("TSLA", "Tesla Inc.", "Consumer Discretionary", "Automobiles"),
        
        # Finance
        ("JPM", "JPMorgan Chase & Co.", "Financials", "Banks"),
        ("BAC", "Bank of America Corp.", "Financials", "Banks"),
        ("WFC", "Wells Fargo & Company", "Financials", "Banks"),
        ("GS", "Goldman Sachs Group Inc.", "Financials", "Investment Banking & Brokerage"),
        ("MS", "Morgan Stanley", "Financials", "Investment Banking & Brokerage"),
        ("AXP", "American Express Company", "Financials", "Consumer Finance"),
        ("V", "Visa Inc.", "Information Technology", "Data Processing & Outsourced Services"),
        ("MA", "Mastercard Incorporated", "Information Technology", "Data Processing & Outsourced Services"),
        
        # Healthcare
        ("JNJ", "Johnson & Johnson", "Health Care", "Pharmaceuticals"),
        ("PFE", "Pfizer Inc.", "Health Care", "Pharmaceuticals"),
        ("UNH", "UnitedHealth Group Incorporated", "Health Care", "Health Care Plans"),
        ("ABBV", "AbbVie Inc.", "Health Care", "Biotechnology"),
        
        # Industrial
        ("BA", "Boeing Company", "Industrials", "Aerospace & Defense"),
        ("CAT", "Caterpillar Inc.", "Industrials", "Construction & Farm Machinery & Heavy Trucks"),
        ("GE", "General Electric Company", "Industrials", "Industrial Conglomerates"),
        
        # Consumer
        ("KO", "Coca-Cola Company", "Consumer Staples", "Soft Drinks"),
        ("PEP", "PepsiCo Inc.", "Consumer Staples", "Soft Drinks"),
        ("WMT", "Walmart Inc.", "Consumer Staples", "Hypermarkets & Super Centers"),
        ("HD", "Home Depot Inc.", "Consumer Discretionary", "Home Improvement Retail"),
        ("MCD", "McDonald's Corporation", "Consumer Discretionary", "Restaurants"),
    ]
    
    df = pd.DataFrame(fallback_data, columns=["Symbol", "Security", "GICS Sector", "GICS Sub‑Industry"])
    print(f"✅  Using fallback dataset with {len(df)} major tickers")
    return df


def filter_tickers_by_industry(industry: str,
                               constituents: pd.DataFrame) -> List[str]:
    """
    Return a list of ticker symbols whose sector **or** sub‑industry contains the
    supplied industry string (case‑insensitive).  The function works even if the
    DataFrame column names differ from the original script because we locate
    the relevant columns dynamically.
    """
    # ------------------------------------------------------------------- #
    # Normalise the user query
    # ------------------------------------------------------------------- #
    industry = industry.lower().strip()
    if not industry:
        raise ValueError("Industry string must not be empty")

    # ------------------------------------------------------------------- #
    # Find the column names that hold sector & sub‑industry information.
    # (We are tolerant of variations like "Sector", "GICS Sector",
    # "Industry", "Sub‑Industry", etc.)
    # ------------------------------------------------------------------- #
    sector_col = next(
        (c for c in constituents.columns if "sector" in c.lower()),
        None
    )
    subind_col = next(
        (c for c in constituents.columns
         if "sub" in c.lower() and "industry" in c.lower()),
        None
    )

    # If we cannot locate either column, raise a clear error.
    if sector_col is None and subind_col is None:
        raise KeyError(
            "Constituents DataFrame does not contain a sector or sub‑industry column. "
            f"Available columns: {list(constituents.columns)}"
        )

    # ------------------------------------------------------------------- #
    # Build a Boolean mask: match either sector or sub‑industry
    # ------------------------------------------------------------------- #
    mask = pd.Series(False, index=constituents.index)

    if sector_col:
        mask |= constituents[sector_col].fillna("").astype(str).str.lower().str.contains(industry, na=False)

    if subind_col:
        mask |= constituents[subind_col].fillna("").astype(str).str.lower().str.contains(industry, na=False)

    # ------------------------------------------------------------------- #
    # Pull the symbols that satisfy the mask
    # ------------------------------------------------------------------- #
    tickers = constituents.loc[mask, "Symbol"].tolist()

    # ------------------------------------------------------------------- #
    # Fallback – try to match the *company name* (Security or Name column)
    # ------------------------------------------------------------------- #
    if not tickers:
        name_col = next(
            (c for c in constituents.columns if c.lower() in {"security", "name"}),
            None
        )
        if name_col:
            mask_name = constituents[name_col].fillna("").astype(str).str.lower().str.contains(industry, na=False)
            tickers = constituents.loc[mask_name, "Symbol"].tolist()

    return tickers


def fetch_stock_data(ticker: str,
                     start: dt.date,
                     end: dt.date) -> pd.DataFrame:
    """
    Download OHLCV for **exactly one** ticker.
    Returns a DataFrame whose index is a true ``pd.DatetimeIndex`` named 'date'.
    
    FIXED: Properly handles MultiIndex columns that yfinance sometimes returns.
    """
    # ------------------------------------------------------------------- #
    # Pull raw data from yfinance (single string → single‑ticker DF)
    # ------------------------------------------------------------------- #
    raw = yf.download(ticker,
                      start=start,
                      end=end,
                      progress=False,
                      auto_adjust=False)          # keep raw Close / Adj Close separate

    if raw.empty:
        return pd.DataFrame()                     # caller will skip this ticker

    # ------------------------------------------------------------------- #
    # FIXED: Handle MultiIndex columns that yfinance sometimes creates
    # ------------------------------------------------------------------- #
    if isinstance(raw.columns, pd.MultiIndex):
        # If we have a MultiIndex, flatten it by taking the first level
        # This happens when yfinance returns data in a multi-ticker format
        raw.columns = raw.columns.get_level_values(0)
    
    # ------------------------------------------------------------------- #
    # Keep only the columns we really need and ensure a clean index
    # ------------------------------------------------------------------- #
    price_cols = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
    # Only select columns that actually exist in the DataFrame
    available_cols = [col for col in price_cols if col in raw.columns]
    raw = raw[available_cols]

    # yfinance returns a ``DatetimeIndex`` already – keep it as‑is
    raw.index = pd.to_datetime(raw.index)          # guarantee Timestamp objects
    raw.index.name = "date"                        # set a friendly name
    return raw

def merge_price_and_sentiment(price_df: pd.DataFrame,
                             sentiment_df: pd.DataFrame) -> pd.DataFrame:
    """
    Join daily price data with daily sentiment scores.
    Guarantees:
        * a single‑level column called ``date`` on both tables,
        * a ``datetime64[ns]`` (``DatetimeIndex``) after the merge.
    
    FIXED: Better handling of MultiIndex and column level mismatches.
    """
    # ------------------------------------------------------------------- #
    # 1️⃣  Normalise the price table – make sure a column named ``date`` exists
    # ------------------------------------------------------------------- #
    price = price_df.copy()
    
    # FIXED: Handle MultiIndex columns if they exist
    if isinstance(price.columns, pd.MultiIndex):
        price.columns = price.columns.get_level_values(0)

    # If the index is already a datetime index, move it to a column called 'date'
    if price.index.name == "date" and isinstance(price.index, pd.DatetimeIndex):
        price = price.reset_index()                       # creates a 'date' column
    elif price.index.name is None or not isinstance(price.index, pd.DatetimeIndex):
        # Fallback – just reset whatever index we have
        price = price.reset_index()
        # If the resulting column is not called 'date', rename it
        if "date" not in price.columns:
            price = price.rename(columns={price.columns[0]: "date"})

    # Ensure the date column is truly datetime
    price["date"] = pd.to_datetime(price["date"])

    # ------------------------------------------------------------------- #
    # 2️⃣  Normalise the sentiment table – it must also have a datetime column named 'date'
    # ------------------------------------------------------------------- #
    sentiment = sentiment_df.copy()
    if "date" not in sentiment.columns:
        # Sentiment might already be indexed by date
        if sentiment.index.name == "date" and isinstance(sentiment.index, pd.DatetimeIndex):
            sentiment = sentiment.reset_index()
        else:
            raise KeyError("Sentiment DataFrame must contain a 'date' column (or be indexed by date).")
    sentiment["date"] = pd.to_datetime(sentiment["date"])

    # ------------------------------------------------------------------- #
    # 3️⃣  Simple left‑join on the date column
    # ------------------------------------------------------------------- #
    merged = pd.merge(price,
                      sentiment,
                      how="left",
                      on="date",
                      suffixes=("", "_sent"))

    # ------------------------------------------------------------------- #
    # 4️⃣  Fill missing sentiment values – a missing day ⇢ neutral (score = 0)
    # ------------------------------------------------------------------- #
    sentiment_cols = ["pos_count", "neg_count", "neu_count", "total", "sentiment_score"]
    for col in sentiment_cols:
        if col in merged.columns:
            merged[col] = merged[col].fillna(0)

    # ------------------------------------------------------------------- #
    # 5️⃣  Put the date back as the index (now a proper DatetimeIndex)
    # ------------------------------------------------------------------- #
    merged.set_index("date", inplace=True)

    # Verify the index type – if something went wrong raise a helpful error
    if not isinstance(merged.index, pd.DatetimeIndex):
        raise TypeError("After merging, the index is not a DatetimeIndex. "
                        "Check that both inputs contain proper datetime values.")

    return merged


def fetch_news_articles(query: str,
                        n_articles: int = DEFAULT_NEWS_COUNT) -> pd.DataFrame:
    """
    Pull the most recent `n_articles` news items from NewsAPI that contain the query string.
    Returns a DataFrame with columns: publishedAt (date), title, description, content, url.
    """
    url = "https://newsapi.org/v2/everything"
    params = {
        "q": query,
        "language": "en",
        "pageSize": n_articles,
        "sortBy": "publishedAt",
        "apiKey": NEWSAPI_KEY,
    }
    response = requests.get(url, params=params, timeout=10)
    if response.status_code != 200:
        raise RuntimeError(f"NewsAPI request failed: {response.status_code} – {response.text}")
    raw = response.json()
    articles = raw.get("articles", [])
    if not articles:
        print(f"⚠️  No news articles found for query '{query}'.")
        return pd.DataFrame()
    df = pd.DataFrame(articles)
    # Keep important fields and parse date
    df = df[["publishedAt", "title", "description", "content", "url"]].copy()
    df["publishedAt"] = pd.to_datetime(df["publishedAt"]).dt.date
    # Drop rows with missing content (fallback to title+description)
    df["content"] = df["content"].fillna(df["title"]).fillna(df["description"])
    return df


def compute_sentiment(df_articles: pd.DataFrame) -> pd.DataFrame:
    """
    Run VADER sentiment analysis on the article `content`.
    Returns a DataFrame with columns: date, compound (score), sentiment_label.
    """
    sia = SentimentIntensityAnalyzer()
    scores = []
    for _, row in df_articles.iterrows():
        text = str(row["content"])
        compound = sia.polarity_scores(text)["compound"]
        if compound >= POSITIVE_THRESH:
            label = "positive"
        elif compound <= NEGATIVE_THRESH:
            label = "negative"
        else:
            label = "neutral"
        scores.append({"date": row["publishedAt"],
                       "compound": compound,
                       "sentiment": label})
    df_sent = pd.DataFrame(scores)
    # Aggregate to daily average compound score (also count sentiments)
    daily = df_sent.groupby("date").agg(
        avg_compound=("compound", "mean"),
        pos_count=("sentiment", lambda x: (x == "positive").sum()),
        neg_count=("sentiment", lambda x: (x == "negative").sum()),
        neu_count=("sentiment", lambda x: (x == "neutral").sum()),
        total=("sentiment", "count")
    )
    daily["sentiment_score"] = daily["avg_compound"]   # alias used later
    daily = daily.drop(columns=["avg_compound"])
    daily = daily.reset_index()
    return daily   # columns: date, pos_count, neg_count, neu_count, total, sentiment_score

def train_sarimax_and_forecast(merged_df: pd.DataFrame,
                               steps: int = 7) -> Tuple[pd.Series, sarimax.SARIMAXResults]:
    """
    Fit a SARIMAX(1,1,1) model on the Adj Close series using the daily
    ``sentiment_score`` as an exogenous regressor.
    Returns a Series of length ``steps`` containing the predicted Adj Close.
    """
    # --------------------------------------------------------------- #
    # Make absolutely sure we are feeding a datetime index to the model
    # --------------------------------------------------------------- #
    if not isinstance(merged_df.index, pd.DatetimeIndex):
        merged_df = merged_df.copy()
        merged_df.index = pd.to_datetime(merged_df.index)

    # --------------------------------------------------------------- #
    # Endogenous & exogenous variables
    # --------------------------------------------------------------- #
    y = merged_df["Adj Close"]
    exog = merged_df[["sentiment_score"]]

    if len(y) < 30:
        raise ValueError("Not enough observations for SARIMAX (need ≥30 days).")

    # --------------------------------------------------------------- #
    # Fit SARIMAX
    # --------------------------------------------------------------- #
    model = SARIMAX(y,
                    exog=exog,
                    order=(1, 1, 1),
                    enforce_stationarity=False,
                    enforce_invertibility=False)
    result = model.fit(disp=False)

    # --------------------------------------------------------------- #
    # Build the future exogenous dataframe (use the most‑recent sentiment)
    # --------------------------------------------------------------- #
    last_sentiment = exog["sentiment_score"].iloc[-1]
    future_dates = pd.date_range(start=merged_df.index[-1] + dt.timedelta(days=1),
                                periods=steps, freq="D")
    future_exog = pd.DataFrame(
        {"sentiment_score": np.full(steps, last_sentiment)},
        index=future_dates
    )

    # --------------------------------------------------------------- #
    # Forecast – we explicitly pass the future exog index that we just built
    # --------------------------------------------------------------- #
    forecast_res = result.get_forecast(steps=steps, exog=future_exog)
    forecast_series = forecast_res.predicted_mean

    # FIXED: Handle case where forecast_series might not have a DatetimeIndex
    if not isinstance(forecast_series.index, pd.DatetimeIndex):
        # If the index is a RangeIndex, create proper dates manually
        forecast_series.index = future_dates
    
    # Convert DatetimeIndex to date objects for pretty printing
    if hasattr(forecast_series.index, 'date'):
        forecast_series.index = forecast_series.index.date
    else:
        # Fallback: convert to datetime first, then to date
        forecast_series.index = pd.to_datetime(forecast_series.index).date
    
    return forecast_series, result


def format_forecast_output(ticker: str,
                           forecast: pd.Series) -> pd.DataFrame:
    """
    Turn a series of forecasts into a nice DataFrame with ticker column.
    """
    df = forecast.reset_index()
    df.columns = ["date", "pred_adj_close"]
    df["ticker"] = ticker
    return df[["ticker", "date", "pred_adj_close"]]

def save_results_to_file(results_df: pd.DataFrame,
                        industry: str,
                        base_name: str = "stock_forecast",
                        output_dir: str = "report") -> str:
    """
    Save the results DataFrame to a CSV file in a subdirectory with configurable naming.
    Format: One row per ticker with columns for each forecast day plus 7-day change.
    
    Parameters:
    - results_df: DataFrame containing the forecast results
    - industry: Industry name used in the analysis (for filename)
    - base_name: Base name for the output file (default: "stock_forecast")
    - output_dir: Subdirectory name where files will be saved (default: "report")
    
    Returns:
    - str: Full path to the saved file
    """
    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Generate filename with current date and sanitized industry name
    current_date = dt.date.today().strftime("%Y%m%d")
    # Sanitize industry name for filename (replace spaces and special chars)
    clean_industry = "".join(c for c in industry if c.isalnum() or c in (' ', '-', '_')).rstrip()
    clean_industry = clean_industry.replace(' ', '_').lower()
    
    filename = f"{base_name}_{clean_industry}_{current_date}.csv"
    filepath = os.path.join(output_dir, filename)
    
    # Transform the data: pivot to have one row per ticker with columns for each day
    pivot_df = results_df.pivot(index='ticker', columns='date', values='pred_adj_close')
    
    # Sort columns (dates) in chronological order
    pivot_df = pivot_df.reindex(sorted(pivot_df.columns), axis=1)
    
    # Keep the actual dates as column names, but format them nicely
    # Convert date objects to strings in YYYY-MM-DD format for better readability
    formatted_dates = []
    for date_col in pivot_df.columns:
        if hasattr(date_col, 'strftime'):
            # If it's already a date object
            formatted_dates.append(date_col.strftime('%Y-%m-%d'))
        else:
            # If it's a string, try to parse and reformat
            try:
                parsed_date = pd.to_datetime(date_col).strftime('%Y-%m-%d')
                formatted_dates.append(parsed_date)
            except:
                # Fallback: use as-is
                formatted_dates.append(str(date_col))
    
    pivot_df.columns = formatted_dates
    day_columns = formatted_dates
    
    # Calculate the 7-day change (Day_7 - Day_1)
    if len(day_columns) >= 2:
        pivot_df['7_Day_Change'] = pivot_df[day_columns[-1]] - pivot_df[day_columns[0]]
        pivot_df['7_Day_Change_Pct'] = ((pivot_df[day_columns[-1]] - pivot_df[day_columns[0]]) / pivot_df[day_columns[0]] * 100).round(2)
    else:
        pivot_df['7_Day_Change'] = 0
        pivot_df['7_Day_Change_Pct'] = 0
    
    # Round all numeric values to 2 decimal places
    numeric_cols = pivot_df.select_dtypes(include=[np.number]).columns
    pivot_df[numeric_cols] = pivot_df[numeric_cols].round(2)
    
    # Reset index to make ticker a column
    pivot_df = pivot_df.reset_index()
    
    # Save the pivoted DataFrame to CSV
    pivot_df.to_csv(filepath, index=False)
    
    # Also save a summary report with metadata
    summary_filename = f"{base_name}_{clean_industry}_{current_date}_summary.txt"
    summary_filepath = os.path.join(output_dir, summary_filename)
    
    with open(summary_filepath, 'w') as f:
        f.write(f"Stock Forecast Analysis Report\n")
        f.write(f"=" * 40 + "\n\n")
        f.write(f"Analysis Date: {dt.date.today().strftime('%Y-%m-%d')}\n")
        f.write(f"Industry: {industry}\n")
        f.write(f"Total Tickers Analyzed: {pivot_df.shape[0]}\n")
        f.write(f"Forecast Period: {FORECAST_DAYS} days\n\n")
        
        f.write(f"Tickers in Analysis:\n")
        for ticker in sorted(pivot_df['ticker']):
            f.write(f"  - {ticker}\n")
        
        f.write(f"\nData Format:\n")
        f.write(f"  - One row per ticker\n")
        f.write(f"  - Date columns: {', '.join(day_columns)}\n")
        f.write(f"  - 7_Day_Change: Absolute dollar change from first to last day\n")
        f.write(f"  - 7_Day_Change_Pct: Percentage change from first to last day\n\n")
        
        # Add some basic statistics
        if not pivot_df.empty and '7_Day_Change_Pct' in pivot_df.columns:
            avg_change = pivot_df['7_Day_Change_Pct'].mean()
            max_gainer = pivot_df.loc[pivot_df['7_Day_Change_Pct'].idxmax()]
            max_loser = pivot_df.loc[pivot_df['7_Day_Change_Pct'].idxmin()]
            
            f.write(f"Forecast Summary:\n")
            f.write(f"  - Average 7-day change: {avg_change:.2f}%\n")
            f.write(f"  - Best performer: {max_gainer['ticker']} ({max_gainer['7_Day_Change_Pct']:.2f}%)\n")
            f.write(f"  - Worst performer: {max_loser['ticker']} ({max_loser['7_Day_Change_Pct']:.2f}%)\n\n")
        
        f.write(f"Files Generated:\n")
        f.write(f"  - Data: {filename}\n")
        f.write(f"  - Summary: {summary_filename}\n")
    
    return filepath



# --------------------------------------------------------------------------- #
# Main execution workflow
# --------------------------------------------------------------------------- #

def main():
    # ------------------------------------------------------------------- #
    # 1️⃣  User inputs
    # ------------------------------------------------------------------- #
    industry = input("Enter the industry to analyse (e.g. technology, auto, finance): ").strip()
    months_str = input("Enter look‑back period in months (e.g. 6 for previous 6 months): ").strip()
    try:
        lookback_months = int(months_str)
        if lookback_months <= 0:
            raise ValueError
    except ValueError:
        print("❌  Invalid number of months. Please enter a positive integer.")
        sys.exit(1)

    news_count_str = input(f"How many recent news articles to fetch? (default {DEFAULT_NEWS_COUNT}): ").strip()
    if news_count_str:
        try:
            news_count = int(news_count_str)
        except ValueError:
            print("❌  Invalid news count – falling back to default.")
            news_count = DEFAULT_NEWS_COUNT
    else:
        news_count = DEFAULT_NEWS_COUNT

    # ------------------------------------------------------------------- #
    # 2️⃣  Determine date range
    # ------------------------------------------------------------------- #
    today = dt.date.today()
    end_date = today
    start_date = today - dt.timedelta(days=lookback_months * 30)  # approximate month = 30 days
    print(f"\n📅  Pulling price data from {start_date} to {end_date} ...")

    # ------------------------------------------------------------------- #
    # 3️⃣  Get tickers for the requested industry
    # ------------------------------------------------------------------- #
    constituents = get_sp500_constituents()
    tickers = filter_tickers_by_industry(industry, constituents)

    if not tickers:
        print(f"❌  No tickers found for industry '{industry}'. Exiting.")
        sys.exit(1)

    # Limit to a manageable number
    if len(tickers) > MAX_TICKERS:
        print(f"🔢  Found {len(tickers)} tickers – limiting to the first {MAX_TICKERS}.")
        tickers = tickers[:MAX_TICKERS]

    print(f"✅  {len(tickers)} tickers will be processed: {', '.join(tickers)}\n")

    # ------------------------------------------------------------------- #
    # 4️⃣  Fetch recent news and compute daily sentiment
    # ------------------------------------------------------------------- #
    print(f"🗞️  Fetching up to {news_count} recent news articles for '{industry}' ...")
    news_df = fetch_news_articles(industry, n_articles=news_count)
    if not news_df.empty:
        sentiment_daily_df = compute_sentiment(news_df)
        print(f"   → {len(sentiment_daily_df)} unique dates with sentiment data.\n")
    else:
        # No news → create a neutral sentiment frame (all zeros)
        sentiment_daily_df = pd.DataFrame(columns=[
            "date", "pos_count", "neg_count", "neu_count", "total", "sentiment_score"
        ])
        print("⚠️  No news found – proceeding with neutral sentiment (score=0).\n")

    # ------------------------------------------------------------------- #
    # 5️⃣  Loop over tickers, download price data, merge sentiment, forecast
    # ------------------------------------------------------------------- #
    all_forecasts = []   # collect DataFrames
    failed_tickers = []

    for ticker in tqdm(tickers, desc="Processing tickers"):
        try:
            price_df = fetch_stock_data(ticker, start=start_date, end=end_date)
            if price_df.empty:
                print(f"⚠️  No price data available for {ticker}")
                failed_tickers.append(ticker)
                continue

            # Merge with sentiment (left join → keep all price rows)
            merged_df = merge_price_and_sentiment(price_df, sentiment_daily_df)

            # If the sentiment series is completely empty (no news), fill with 0
            if merged_df["sentiment_score"].isnull().all():
                merged_df["sentiment_score"] = 0.0

            forecast_series, model_res = train_sarimax_and_forecast(merged_df, steps=FORECAST_DAYS)
            
            # Format the forecast and stash it
            pred_df = format_forecast_output(ticker, forecast_series)
            all_forecasts.append(pred_df)
            
        except Exception as exc:
            print(f"⚠️  Processing failed for {ticker}: {exc}")
            failed_tickers.append(ticker)
            continue

    # ------------------------------------------------------------------- #
    # 6️⃣  Present results
    # ------------------------------------------------------------------- #
    if all_forecasts:
        final_df = pd.concat(all_forecasts, ignore_index=True)
        final_df.sort_values(["ticker", "date"], inplace=True)

        # Save results to file
        try:
            output_file = save_results_to_file(final_df, industry)
            print(f"\n💾  Results saved to: {output_file}")
            print(f"📁  Summary report also saved in the same directory.")
        except Exception as e:
            print(f"⚠️  Could not save results to file: {e}")
        
        pd.set_option('display.max_rows', None)
        print("\n🔮  7‑day predicted adjusted close prices:\n")
        print(final_df.to_string(index=False))

        
    else:
        print("\n❌  No forecasts were generated.")

    if failed_tickers:
        print("\n⚠️  The following tickers could not be processed:")
        print(", ".join(failed_tickers))

    print("\n🚀  Done!")


def tester():
    # 1️⃣  Pull a single ticker
    price = fetch_stock_data("AAPL", start=dt.date.today() - dt.timedelta(days=180), end=dt.date.today())
    print(price.head())
    print("columns:", price.columns)
    print("index name:", price.index.name)
    
    # 2️⃣  Fake a sentiment frame (two days only)
    sent = pd.DataFrame({
        "date": [price.index[0], price.index[1]],
        "sentiment_score": [0.12, -0.08],
        "pos_count": [1,0],
        "neg_count": [0,1],
        "neu_count": [0,0],
        "total": [1,1]
    })
    print("\nSentiment sample:")
    print(sent)
    
    # 3️⃣  Merge
    merged = merge_price_and_sentiment(price, sent)
    print("\nMerged head:")
    print(merged.head())


# if __name__ == "__main__":
#     main()

In [2]:
# import nltk
# nltk.download('vader_lexicon')

In [3]:
main()

Enter the industry to analyse (e.g. technology, auto, finance):  ai
Enter look‑back period in months (e.g. 6 for previous 6 months):  6
How many recent news articles to fetch? (default 20):  



📅  Pulling price data from 2025-08-27 to 2026-02-23 ...
🔍  Trying Wikipedia...
⚠️  Wikipedia failed: HTTP Error 403: Forbidden
🔍  Trying GitHub CSV (datasets)...
✅  Successfully loaded 503 tickers from GitHub CSV (datasets)
🔢  Found 43 tickers – limiting to the first 20.
✅  20 tickers will be processed: AMZN, AZO, BALL, BBY, CHRW, KMX, COST, CSX, DAL, DG, DLTR, EBAY, EA, EXPD, FRT, FDX, HD, KIM, KR, LYV

🗞️  Fetching up to 20 recent news articles for 'ai' ...
   → 1 unique dates with sentiment data.



Processing tickers:   0%|                                                                                                                                                                 | 0/20 [00:00<?, ?it/s]/home/javastarchild/anaconda3/envs/mcp-class_3_13/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
Processing tickers:   5%|███████▋                                                                                                                                                 | 1/20 [00:00<00:08,  2.19it/s]/home/javastarchild/anaconda3/envs/mcp-class_3_13/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  ret


💾  Results saved to: report/stock_forecast_ai_20260223.csv
📁  Summary report also saved in the same directory.

🔮  7‑day predicted adjusted close prices:

ticker       date  pred_adj_close
  AMZN 2026-02-21      210.862414
  AMZN 2026-02-22      210.506071
  AMZN 2026-02-23      210.674835
  AMZN 2026-02-24      210.594908
  AMZN 2026-02-25      210.632762
  AMZN 2026-02-26      210.614834
  AMZN 2026-02-27      210.623325
   AZO 2026-02-21     3748.413212
   AZO 2026-02-22     3748.360422
   AZO 2026-02-23     3748.334331
   AZO 2026-02-24     3748.321435
   AZO 2026-02-25     3748.315061
   AZO 2026-02-26     3748.311910
   AZO 2026-02-27     3748.310353
  BALL 2026-02-21       66.610174
  BALL 2026-02-22       66.649796
  BALL 2026-02-23       66.669378
  BALL 2026-02-24       66.679055
  BALL 2026-02-25       66.683837
  BALL 2026-02-26       66.686200
  BALL 2026-02-27       66.687368
   BBY 2026-02-21       64.884054
   BBY 2026-02-22       64.781136
   BBY 2026-02-23       64.8